# 20 counties (g4)

A county for every project in the newest contracted lists. What you should see: at least 95% of contracted MW placed; how much by name (printed in an ECP results publication), by connection node (where the node's other projects are) or by the node's substation in OpenStreetMap; the projects left, by name. Then check a few of the largest by hand against their planning reference or a map.

In [ ]:
import subprocess

from iegrid import store
from iegrid.checks import county_coverage, county_sample, unplaced

snap = store.latest_snapshot()
sha = subprocess.run(["git", "-C", str(store.ROOT), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, check=True).stdout.strip()
con = store.connect_read_only(snap)
print(f"snapshot {snap.name}  git {sha}")

## done-when: at least 95% of contracted MW has a county

In [ ]:
cov = county_coverage(con)
for basis, mw in sorted(cov.by_basis.items(), key=lambda kv: -kv[1]):
    print(f"{basis:9} {float(mw) / 1000:6.2f} GW  {float(mw / cov.total_mw):6.1%}")
print(f"\nbuild {cov.build_id}: {float(cov.placed_mw) / 1000:.2f} of {float(cov.total_mw) / 1000:.2f} GW placed, {cov.share:.1%}")
assert cov.share >= 0.95, f"only {cov.share:.1%} of contracted MW has a county"


## the rest, by name

In [ ]:
print(unplaced(con))

## hand check: the largest placed projects and why

In [ ]:
print(county_sample(con))

## exhibit: where it is stuck

In [ ]:
from iegrid import exhibits

fig = exhibits.contracted_by_county(con)
exhibits.save(fig, "g4_contracted_by_county")